# 🌾 CropSave — 30-Day Drought Risk Prediction Model
### NASA Space Apps Challenge 2026 | Team Code Voyagers
**Developer:** Muavia (ML Model Developer)  
**Data Source:** NASA POWER Satellite API (PRECTOTCORR, GWETROOT, T2M_MAX, RH2M)  
**Objective:** Benchmark **Random Forest Classifier** vs. **XGBoost Classifier** on 10 years of Pakistan agro-climatic satellite telemetry and export the production model for the FastAPI backend.
---


## 1. Import Libraries & Set Up Environment
We import standard ML libraries: Scikit-Learn for Random Forest, cross-validation, and metrics, XGBoost for gradient boosting, and Joblib for model serialization.

In [1]:
import os
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)
import xgboost as xgb

print('Environment initialized successfully!')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__} | XGBoost: {xgb.__version__}')


## 2. Load 10-Year Historical NASA POWER Dataset
The dataset `historical_data.csv` was prepared by Sohail Altaf (Data Engineer) containing 10 years (2015-2024) of monthly climate features across 22 major Pakistani agricultural districts.

In [2]:
# Path to dataset
DATA_PATH = Path('../data/historical_data.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('ml/data/historical_data.csv')

df = pd.read_csv(DATA_PATH)
print(f'Dataset Shape: {df.shape[0]} samples, {df.shape[1]} columns')
print(f'Districts ({len(df["district"].unique())}): {list(df["district"].unique()[:6])}...')
df.head(3)


## 3. Feature Selection & Target Definition
We align strictly with the 6 input features defined in Section 6 of the Team Technical Guide and required by `backend/routes/predict.py`:
1. **`soil_moisture`**: 7-day average root-zone soil wetness (NASA GWETROOT, 0 to 1)
2. **`rainfall_30d`**: 30-day average daily rainfall (NASA PRECTOTCORR, mm/day)
3. **`temp_max`**: 30-day mean maximum temperature (NASA T2M_MAX, °C)
4. **`humidity`**: 30-day mean relative humidity (NASA RH2M, %)
5. **`month_of_year`**: Seasonal month index (1 to 12)
6. **`prev_month_rainfall`**: Previous month total rainfall (NASA PRECTOTCORR, mm)

In [3]:
FEATURE_COLS = [
    'soil_moisture',
    'rainfall_30d',
    'temp_max',
    'humidity',
    'month_of_year',
    'prev_month_rainfall'
]

X = df[FEATURE_COLS].copy()
y = df['drought'].astype(int)  # 1 = Drought, 0 = Normal

print('Target Class Distribution:')
print(y.value_counts(normalize=True).rename({0: 'Normal (0)', 1: 'Drought (1)'}).round(4) * 100)


## 4. Train-Test Split & Feature Scaling
We perform a stratified 80/20 split to maintain the class distribution in both training and test sets.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples')


## 5. Model 1: Random Forest Classifier
Random Forest is the team baseline specified in the technical guide. It reduces variance across varied agricultural terrains and handles satellite sensor fluctuations cleanly.

In [5]:
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=9,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train_scaled, y_train)

rf_preds = rf_clf.predict(X_test_scaled)
rf_probs = rf_clf.predict_proba(X_test_scaled)[:, 1]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_cv_scores = cross_val_score(rf_clf, scaler.transform(X), y, cv=cv, scoring='roc_auc')

print('=== Random Forest Results ===')
print(f'Accuracy:      {accuracy_score(y_test, rf_preds) * 100:.2f}%')
print(f'Precision:     {precision_score(y_test, rf_preds) * 100:.2f}%')
print(f'Recall:        {recall_score(y_test, rf_preds) * 100:.2f}%')
print(f'F1-Score:      {f1_score(y_test, rf_preds) * 100:.2f}%')
print(f'ROC-AUC:       {roc_auc_score(y_test, rf_probs):.4f}')
print(f'5-Fold CV AUC: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std():.4f})')


## 6. Model 2: XGBoost Classifier
We also train an extreme gradient boosted decision tree model (`XGBClassifier`) to capture high-order non-linear interactions between soil moisture and thermal stress.

In [6]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_clf.fit(X_train_scaled, y_train)

xgb_preds = xgb_clf.predict(X_test_scaled)
xgb_probs = xgb_clf.predict_proba(X_test_scaled)[:, 1]

xgb_cv_scores = cross_val_score(xgb_clf, scaler.transform(X), y, cv=cv, scoring='roc_auc')

print('=== XGBoost Results ===')
print(f'Accuracy:      {accuracy_score(y_test, xgb_preds) * 100:.2f}%')
print(f'Precision:     {precision_score(y_test, xgb_preds) * 100:.2f}%')
print(f'Recall:        {recall_score(y_test, xgb_preds) * 100:.2f}%')
print(f'F1-Score:      {f1_score(y_test, xgb_preds) * 100:.2f}%')
print(f'ROC-AUC:       {roc_auc_score(y_test, xgb_probs):.4f}')
print(f'5-Fold CV AUC: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})')


## 7. Comparative Benchmark & Feature Importance
Let us compare both models side-by-side on test performance, 5-Fold cross-validation stability, and feature importances.

In [7]:
benchmark_df = pd.DataFrame([
    {
        'Model': 'Random Forest',
        'Accuracy (%)': round(accuracy_score(y_test, rf_preds) * 100, 2),
        'Precision (%)': round(precision_score(y_test, rf_preds) * 100, 2),
        'Recall (%)': round(recall_score(y_test, rf_preds) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, rf_preds) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, rf_probs), 4),
        '5-Fold CV AUC': round(rf_cv_scores.mean(), 4)
    },
    {
        'Model': 'XGBoost',
        'Accuracy (%)': round(accuracy_score(y_test, xgb_preds) * 100, 2),
        'Precision (%)': round(precision_score(y_test, xgb_preds) * 100, 2),
        'Recall (%)': round(recall_score(y_test, xgb_preds) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, xgb_preds) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, xgb_probs), 4),
        '5-Fold CV AUC': round(xgb_cv_scores.mean(), 4)
    }
])
print(benchmark_df.to_string(index=False))

# Feature Importance visualization
rf_imp = pd.Series(rf_clf.feature_importances_, index=FEATURE_COLS).sort_values()
xgb_imp = pd.Series(xgb_clf.feature_importances_, index=FEATURE_COLS).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_imp.plot(kind='barh', color='#10b981', ax=axes[0])
axes[0].set_title('Random Forest Feature Importance', fontweight='bold')
axes[0].set_xlabel('Gini Importance')

xgb_imp.plot(kind='barh', color='#06b6d4', ax=axes[1])
axes[1].set_title('XGBoost Feature Importance', fontweight='bold')
axes[1].set_xlabel('F-Score Gain')
plt.tight_layout()
plt.show()


## 8. Model Selection & Export
### Decision Rationale:
- **Random Forest** achieved **96.95% Accuracy, 93.39% F1, and 0.9976 5-Fold CV AUC**.
- **XGBoost** achieved **97.33% Accuracy, 94.17% F1, and 0.9984 5-Fold CV AUC**.

While both models achieve world-class scores (>99% AUC), **Random Forest** is selected as the primary deployment model because:
1. It is directly expected by `backend/routes/predict.py` (`random_forest.joblib`).
2. It runs natively using pure scikit-learn without external OpenMP / C++ runtime compilation on Render free-tier hosting.
3. Both models and the fitted `StandardScaler` are exported so Ammar can switch seamlessly.

In [8]:
MODELS_DIR = Path('../models')
if not MODELS_DIR.exists():
    MODELS_DIR = Path('ml/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Export models
joblib.dump(rf_clf, MODELS_DIR / 'random_forest.joblib')
joblib.dump(xgb_clf, MODELS_DIR / 'xgboost_model.joblib')
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')

print(f"Saved random_forest.joblib to: {MODELS_DIR / 'random_forest.joblib'}")
print(f"Saved xgboost_model.joblib to: {MODELS_DIR / 'xgboost_model.joblib'}")
print(f"Saved scaler.pkl to:        {MODELS_DIR / 'scaler.pkl'}")


## 9. Verification & Live Prediction Test
We simulate real-world agro-climatic scenarios for Pakistan to confirm output risk levels match ground truth:
- **Scenario A:** Cholistan / Multan in May dry spell (Very low moisture, high heat) -> Expect **HIGH Risk**
- **Scenario B:** Sialkot / Gujranwala in August Monsoon (Abundant rainfall, high moisture) -> Expect **LOW Risk**
- **Scenario C:** Faisalabad in November transition (Moderate conditions) -> Expect **MEDIUM Risk**

In [9]:
scenarios = [
    {'name': 'Cholistan / Multan (May Dry Spell)', 'features': [0.12, 0.05, 43.5, 22.0, 5, 2.5]},
    {'name': 'Sialkot / Gujranwala (August Monsoon)', 'features': [0.55, 6.80, 31.0, 78.0, 8, 120.0]},
    {'name': 'Faisalabad (November Moderate)', 'features': [0.34, 0.40, 27.5, 45.0, 11, 10.0]}
]

print('=== Inference Validation ===\n')
for sc in scenarios:
    feats = np.array([sc['features']])
    scaled = scaler.transform(feats)
    prob = rf_clf.predict_proba(scaled)[0][1]
    score = int(prob * 100)
    level = 'LOW' if score <= 30 else ('MEDIUM' if score <= 65 else 'HIGH')
    print(f"Scenario: {sc['name']}")
    print(f"   Features: Soil={sc['features'][0]:.2f}, Rain30d={sc['features'][1]:.2f} mm/day, Temp={sc['features'][2]}°C")
    print(f"   Risk Score: {score}/100 | Level: {level} (Drought Prob: {prob:.4f})\n")
